# Chest X-ray Pneumonia Classification (PyTorch)

This notebook implements a deep learning pipeline to classify chest X-ray images into **NORMAL** and **PNEUMONIA**.

## Reproducibility and Runtime Notes
- The dataset is downloaded at execution time from Kaggle.
- Credentials are intentionally placeholders (`xxxxxx`) for safe sharing.
- `QUICK_MODE=True` is the default for faster Colab execution.
- Set `QUICK_MODE=False` to scale to the full dataset.

In [ ]:
# Environment setup and imports
import os
import json
import time
import random
import zipfile
import shutil
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
)

warnings.filterwarnings("ignore")

# Install missing dependencies when running in Colab
if "google.colab" in str(get_ipython()):
    !pip -q install kaggle scikit-learn seaborn opencv-python

In [ ]:
# Global configuration
@dataclass
class Config:
    dataset_slug: str = "yusufmurtaza01/chest-xray-pneumonia-balanced-dataset"
    kaggle_username: str = "xxxxxx"
    kaggle_key: str = "xxxxxx"

    seed: int = 42
    quick_mode: bool = True
    quick_cap_per_class_train: int = 1200
    quick_cap_per_class_val: int = 250
    quick_cap_per_class_test: int = 250

    image_size: int = 224
    batch_size: int = 32
    num_workers: int = 2

    max_epochs: int = 8
    patience: int = 3
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4

    # Model selection budget (3-6 recommended)
    n_trials: int = 4

    train_size: float = 0.70
    val_size: float = 0.15
    test_size: float = 0.15

    root_dir: str = "./artifacts"

cfg = Config()
assert abs((cfg.train_size + cfg.val_size + cfg.test_size) - 1.0) < 1e-9

artifacts_dir = Path(cfg.root_dir)
data_dir = artifacts_dir / "data"
models_dir = artifacts_dir / "models"
reports_dir = artifacts_dir / "reports"
for d in [artifacts_dir, data_dir, models_dir, reports_dir]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Config:")
print(json.dumps(asdict(cfg), indent=2))

In [ ]:
# Reproducibility helpers

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)
print("Seed fixed to", cfg.seed)

In [ ]:
# Kaggle authentication and dataset download

def download_dataset_from_kaggle(cfg: Config, download_dir: Path) -> Path:
    os.environ["KAGGLE_USERNAME"] = cfg.kaggle_username
    os.environ["KAGGLE_KEY"] = cfg.kaggle_key

    zip_target = download_dir / "dataset.zip"
    extract_dir = download_dir / "extracted"

    if extract_dir.exists() and any(extract_dir.rglob("*.jpg")):
        print("Dataset already extracted. Reusing local copy.")
        return extract_dir

    download_dir.mkdir(parents=True, exist_ok=True)

    cmd = f"kaggle datasets download -d {cfg.dataset_slug} -p {download_dir} -f ''"
    # Kaggle CLI writes with dataset-derived zip name. We invoke generic download first.
    print("Downloading dataset from Kaggle...")
    status = os.system(f"kaggle datasets download -d {cfg.dataset_slug} -p {download_dir}")
    if status != 0:
        raise RuntimeError(
            "Kaggle download failed. Check credentials or use kaggle.json authentication."
        )

    zip_files = list(download_dir.glob("*.zip"))
    if not zip_files:
        raise FileNotFoundError("No zip file found after Kaggle download.")

    latest_zip = max(zip_files, key=lambda p: p.stat().st_mtime)
    if zip_target.exists():
        zip_target.unlink()
    shutil.copy(latest_zip, zip_target)

    print("Extracting dataset...")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_target, "r") as zf:
        zf.extractall(extract_dir)

    print("Dataset ready at", extract_dir)
    return extract_dir

extract_root = download_dataset_from_kaggle(cfg, data_dir)

In [ ]:
# Build a dataframe of image paths and labels
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
LABEL_MAP = {"normal": 0, "pneumonia": 1}
INV_LABEL_MAP = {0: "NORMAL", 1: "PNEUMONIA"}


def index_images(root: Path) -> pd.DataFrame:
    rows = []
    for p in root.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in VALID_EXTS:
            continue

        parts = [x.lower() for x in p.parts]
        label_name = None
        if "normal" in parts:
            label_name = "normal"
        elif "pneumonia" in parts:
            label_name = "pneumonia"

        if label_name is None:
            continue

        rows.append({
            "path": str(p),
            "label_str": label_name,
            "label": LABEL_MAP[label_name],
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(
            "No labeled images found. Verify folder names include NORMAL and PNEUMONIA."
        )

    return df.sample(frac=1.0, random_state=cfg.seed).reset_index(drop=True)

all_df = index_images(extract_root)
print("Total images:", len(all_df))
print(all_df["label_str"].value_counts())

In [ ]:
# Stratified split + optional quick-mode subsampling

def stratified_split(df: pd.DataFrame, cfg: Config):
    train_df, temp_df = train_test_split(
        df,
        test_size=(1 - cfg.train_size),
        random_state=cfg.seed,
        stratify=df["label"],
    )

    rel_test_size = cfg.test_size / (cfg.val_size + cfg.test_size)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=rel_test_size,
        random_state=cfg.seed,
        stratify=temp_df["label"],
    )

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


def cap_per_class(df: pd.DataFrame, cap: int, seed: int) -> pd.DataFrame:
    chunks = []
    for lbl, grp in df.groupby("label"):
        if len(grp) > cap:
            chunks.append(grp.sample(n=cap, random_state=seed))
        else:
            chunks.append(grp)
    out = pd.concat(chunks).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

train_df, val_df, test_df = stratified_split(all_df, cfg)

if cfg.quick_mode:
    train_df = cap_per_class(train_df, cfg.quick_cap_per_class_train, cfg.seed)
    val_df = cap_per_class(val_df, cfg.quick_cap_per_class_val, cfg.seed)
    test_df = cap_per_class(test_df, cfg.quick_cap_per_class_test, cfg.seed)

print("Split sizes:", len(train_df), len(val_df), len(test_df))
print("Train class counts:\n", train_df["label_str"].value_counts())
print("Val class counts:\n", val_df["label_str"].value_counts())
print("Test class counts:\n", test_df["label_str"].value_counts())

In [ ]:
# Persist split metadata for reproducibility
split_dir = reports_dir / "splits"
split_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(split_dir / "train.csv", index=False)
val_df.to_csv(split_dir / "val.csv", index=False)
test_df.to_csv(split_dir / "test.csv", index=False)

print("Saved split CSV files to", split_dir)

In [ ]:
# Torch datasets and dataloaders
class XRayDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        label = int(row["label"])

        if self.transform is not None:
            image = self.transform(image)

        return image, label, row["path"]


train_tfms = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = XRayDataset(train_df, transform=train_tfms)
val_ds = XRayDataset(val_df, transform=eval_tfms)
test_ds = XRayDataset(test_df, transform=eval_tfms)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=(DEVICE == "cuda"),
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=(DEVICE == "cuda"),
)
test_loader = DataLoader(
    test_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=(DEVICE == "cuda"),
)

print("DataLoaders ready")
print("train batches:", len(train_loader), "val batches:", len(val_loader), "test batches:", len(test_loader))

In [ ]:
# Data sanity checks

def show_class_distribution(train_df, val_df, test_df):
    dist_df = pd.DataFrame({
        "train": train_df["label_str"].value_counts(),
        "val": val_df["label_str"].value_counts(),
        "test": test_df["label_str"].value_counts(),
    }).fillna(0).astype(int)
    display(dist_df)

    ax = dist_df.T.plot(kind="bar", figsize=(7, 4), rot=0)
    ax.set_title("Class Distribution per Split")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()


def show_sample_grid(dataset, n=8):
    n = min(n, len(dataset))
    idxs = np.random.choice(len(dataset), size=n, replace=False)

    fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(14, 6))
    axes = np.array(axes).reshape(-1)

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    for i, idx in enumerate(idxs):
        x, y, _ = dataset[idx]
        img = x.permute(1, 2, 0).numpy()
        img = (img * std + mean).clip(0, 1)
        axes[i].imshow(img)
        axes[i].set_title(INV_LABEL_MAP[y])
        axes[i].axis("off")

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

show_class_distribution(train_df, val_df, test_df)
show_sample_grid(train_ds, n=8)

In [ ]:
# Model, metrics, and training utilities

def build_model(dropout: float = 0.2, unfreeze_last_block: bool = True):
    model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

    # Freeze all layers first, then optionally unfreeze deeper blocks.
    for p in model.parameters():
        p.requires_grad = False

    if unfreeze_last_block:
        for p in model.features.denseblock4.parameters():
            p.requires_grad = True
        for p in model.features.norm5.parameters():
            p.requires_grad = True

    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, 2)

    return model


def compute_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )

    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float("nan")

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "roc_auc": auc,
    }


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    losses = []
    all_y_true, all_y_pred, all_y_prob = [], [], []

    for xb, yb, _ in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        if is_train:
            loss.backward()
            optimizer.step()

        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = torch.argmax(logits, dim=1)

        losses.append(loss.item())
        all_y_true.extend(yb.detach().cpu().numpy().tolist())
        all_y_pred.extend(preds.detach().cpu().numpy().tolist())
        all_y_prob.extend(probs.detach().cpu().numpy().tolist())

    metrics = compute_metrics(all_y_true, all_y_pred, all_y_prob)
    metrics["loss"] = float(np.mean(losses)) if losses else np.nan
    return metrics


def train_with_early_stopping(model, train_loader, val_loader, lr, weight_decay, max_epochs, patience):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=weight_decay,
    )

    history = []
    best_state = None
    best_val_f1 = -1.0
    epochs_no_improve = 0

    for epoch in range(1, max_epochs + 1):
        train_m = run_epoch(model, train_loader, criterion, optimizer=optimizer)
        val_m = run_epoch(model, val_loader, criterion, optimizer=None)

        row = {
            "epoch": epoch,
            **{f"train_{k}": v for k, v in train_m.items()},
            **{f"val_{k}": v for k, v in val_m.items()},
        }
        history.append(row)

        val_f1 = val_m["f1"]
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        print(
            f"Epoch {epoch:02d} | "
            f"train loss={train_m['loss']:.4f}, f1={train_m['f1']:.4f} | "
            f"val loss={val_m['loss']:.4f}, f1={val_m['f1']:.4f}"
        )

        if epochs_no_improve >= patience:
            print("Early stopping triggered.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_val_f1

In [ ]:
# Lightweight model selection (3-6 trials)
trial_space = [
    {"dropout": 0.1, "lr": 3e-4, "weight_decay": 1e-4, "unfreeze_last_block": True},
    {"dropout": 0.2, "lr": 1e-4, "weight_decay": 1e-4, "unfreeze_last_block": True},
    {"dropout": 0.3, "lr": 3e-4, "weight_decay": 1e-5, "unfreeze_last_block": True},
    {"dropout": 0.2, "lr": 5e-4, "weight_decay": 1e-4, "unfreeze_last_block": False},
    {"dropout": 0.4, "lr": 1e-4, "weight_decay": 1e-5, "unfreeze_last_block": True},
    {"dropout": 0.1, "lr": 5e-4, "weight_decay": 1e-5, "unfreeze_last_block": False},
]

rng = random.Random(cfg.seed)
selected_trials = trial_space[:]
rng.shuffle(selected_trials)
selected_trials = selected_trials[: cfg.n_trials]

trial_rows = []
best_model = None
best_history = None
best_cfg = None
best_val = -1.0

for i, tcfg in enumerate(selected_trials, start=1):
    print("\n" + "=" * 60)
    print(f"Trial {i}/{len(selected_trials)}: {tcfg}")
    seed_everything(cfg.seed + i)

    model = build_model(
        dropout=tcfg["dropout"],
        unfreeze_last_block=tcfg["unfreeze_last_block"],
    )

    t0 = time.time()
    model, hist_df, best_val_f1 = train_with_early_stopping(
        model,
        train_loader,
        val_loader,
        lr=tcfg["lr"],
        weight_decay=tcfg["weight_decay"],
        max_epochs=cfg.max_epochs,
        patience=cfg.patience,
    )
    dt = time.time() - t0

    trial_rows.append({
        "trial": i,
        **tcfg,
        "best_val_f1": best_val_f1,
        "runtime_sec": dt,
        "epochs_ran": int(hist_df["epoch"].max()) if len(hist_df) else 0,
    })

    if best_val_f1 > best_val:
        best_val = best_val_f1
        best_model = model
        best_history = hist_df.copy()
        best_cfg = dict(tcfg)

trial_df = pd.DataFrame(trial_rows).sort_values("best_val_f1", ascending=False)
trial_df.to_csv(reports_dir / "trial_results.csv", index=False)

print("\nBest trial config:", best_cfg)
print("Best validation F1:", best_val)
display(trial_df)

In [ ]:
# Plot best training history
if best_history is not None and not best_history.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(best_history["epoch"], best_history["train_loss"], label="train")
    axes[0].plot(best_history["epoch"], best_history["val_loss"], label="val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(best_history["epoch"], best_history["train_f1"], label="train")
    axes[1].plot(best_history["epoch"], best_history["val_f1"], label="val")
    axes[1].set_title("F1")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    best_history.to_csv(reports_dir / "best_history.csv", index=False)
else:
    print("No training history available.")

In [ ]:
# Final test evaluation helpers

def predict_on_loader(model, loader):
    model.eval()
    y_true, y_pred, y_prob, paths = [], [], [], []

    with torch.no_grad():
        for xb, yb, pths in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)

            y_true.extend(yb.numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())
            y_prob.extend(probs.cpu().numpy().tolist())
            paths.extend(list(pths))

    return np.array(y_true), np.array(y_pred), np.array(y_prob), np.array(paths)


if best_model is None:
    raise RuntimeError("No best model available. Ensure trial section has been executed.")

y_true, y_pred, y_prob, y_paths = predict_on_loader(best_model, test_loader)
test_metrics = compute_metrics(y_true, y_pred, y_prob)

print("Test metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")

pd.DataFrame([test_metrics]).to_csv(reports_dir / "test_metrics.csv", index=False)
torch.save(
    {
        "model_state_dict": best_model.state_dict(),
        "best_trial_config": best_cfg,
        "config": asdict(cfg),
        "test_metrics": test_metrics,
    },
    models_dir / "best_model.pt",
)
print("Saved model checkpoint to", models_dir / "best_model.pt")

In [ ]:
# Confusion matrix, report, and ROC curve
cm = confusion_matrix(y_true, y_pred)
report_txt = classification_report(
    y_true,
    y_pred,
    target_names=[INV_LABEL_MAP[0], INV_LABEL_MAP[1]],
)
print(report_txt)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=[INV_LABEL_MAP[0], INV_LABEL_MAP[1]],
    yticklabels=[INV_LABEL_MAP[0], INV_LABEL_MAP[1]],
    ax=axes[0],
)
axes[0].set_title("Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

fpr, tpr, _ = roc_curve(y_true, y_prob)
axes[1].plot(fpr, tpr, label=f"ROC AUC = {test_metrics['roc_auc']:.3f}")
axes[1].plot([0, 1], [0, 1], linestyle="--")
axes[1].set_title("ROC Curve")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

with open(reports_dir / "classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report_txt)

In [ ]:
# Error analysis: visualize misclassified test images
mis_idx = np.where(y_true != y_pred)[0]
print("Misclassified samples:", len(mis_idx), "/", len(y_true))

if len(mis_idx) > 0:
    show_n = min(8, len(mis_idx))
    chosen = np.random.choice(mis_idx, size=show_n, replace=False)

    fig, axes = plt.subplots(2, (show_n + 1) // 2, figsize=(14, 6))
    axes = np.array(axes).reshape(-1)

    for i, idx in enumerate(chosen):
        img = Image.open(y_paths[idx]).convert("RGB").resize((cfg.image_size, cfg.image_size))
        axes[i].imshow(img)
        axes[i].set_title(
            f"T:{INV_LABEL_MAP[int(y_true[idx])]} / P:{INV_LABEL_MAP[int(y_pred[idx])]}"
        )
        axes[i].axis("off")

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Grad-CAM visualization for explainability

import torch.nn.functional as F


class GradCAM:
    """Hook-free Grad-CAM for DenseNet to avoid inplace ReLU hook conflicts."""

    def __init__(self, model, target_layer=None):
        self.model = model
        self.target_layer = target_layer  # kept for API compatibility

    def generate(self, x, class_idx=1):
        """Generate Grad-CAM heatmap for a given input and class."""
        self.model.eval()

        # Forward pass through DenseNet features (pre-classifier path).
        features = self.model.features(x)

        # Use non-inplace ReLU to avoid autograd view/inplace issues.
        features = torch.relu(features)
        features.retain_grad()

        pooled = F.adaptive_avg_pool2d(features, (1, 1))
        flattened = torch.flatten(pooled, 1)
        logits = self.model.classifier(flattened)

        # Backward pass for target class score.
        self.model.zero_grad(set_to_none=True)
        score = logits[:, class_idx].sum()
        score.backward(retain_graph=False)

        if features.grad is None:
            raise RuntimeError("Grad-CAM gradients were not captured.")

        activation = features.detach()[0]      # (C, H, W)
        gradients = features.grad.detach()[0]  # (C, H, W)

        # Weights = global average pooling of gradients.
        weights = gradients.mean(dim=(1, 2))

        # Weighted sum of activations.
        heatmap = torch.sum(activation * weights.view(-1, 1, 1), dim=0)

        # Normalize to [0, 1].
        heatmap = torch.relu(heatmap)
        heatmap = heatmap / (heatmap.max() + 1e-8)

        return heatmap.cpu().numpy()

    def remove_hooks(self):
        """No-op for compatibility with existing notebook call site."""
        return


def overlay_heatmap(image_tensor, heatmap, alpha=0.4, cmap_name="jet"):
    """Overlay heatmap on original image for visualization."""
    import matplotlib.cm as cm

    # Denormalize image
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_np = image_tensor.permute(1, 2, 0).numpy()
    img_np = (img_np * std + mean).clip(0, 1)

    # Resize heatmap to match image
    heatmap_resized = cv2.resize(heatmap, (img_np.shape[1], img_np.shape[0]))

    # Apply colormap to heatmap
    cmap = cm.get_cmap(cmap_name)
    heatmap_colored = cmap(heatmap_resized)[:, :, :3]  # RGB only

    # Blend image and heatmap
    overlay = (1 - alpha) * img_np + alpha * heatmap_colored

    return overlay


print("Grad-CAM utilities loaded")

In [ ]:
# Generate Grad-CAM visualizations for different classification outcomes

import cv2

# Get the target layer (final convolutional layer of DenseNet)
target_layer = best_model.features.norm5

# Initialize Grad-CAM
gradcam = GradCAM(best_model, target_layer)

# Classify predictions into categories
tn_idx = np.where((y_pred == 0) & (y_true == 0))[0]  # True Negatives (Predicted NORMAL, Actually NORMAL)
tp_idx = np.where((y_pred == 1) & (y_true == 1))[0]  # True Positives (Predicted PNEUMONIA, Actually PNEUMONIA)
fp_idx = np.where((y_pred == 1) & (y_true == 0))[0]  # False Positives (Predicted PNEUMONIA, Actually NORMAL)
fn_idx = np.where((y_pred == 0) & (y_true == 1))[0]  # False Negatives (Predicted NORMAL, Actually PNEUMONIA)

print(f"True Negatives (correct NORMAL): {len(tn_idx)}")
print(f"True Positives (correct PNEUMONIA): {len(tp_idx)}")
print(f"False Positives (wrong -> PNEUMONIA): {len(fp_idx)}")
print(f"False Negatives (wrong -> NORMAL): {len(fn_idx)}")

# Select representative samples from each category
def select_samples(indices, n=2, random_state=42):
    if len(indices) == 0:
        return np.array([])
    n = min(n, len(indices))
    return np.random.RandomState(random_state).choice(indices, size=n, replace=False)

tn_samples = select_samples(tn_idx, n=2)
tp_samples = select_samples(tp_idx, n=2)
fp_samples = select_samples(fp_idx, n=2)
fn_samples = select_samples(fn_idx, n=2)

# Prepare figure with 4 rows (one per category) and 2 columns (original + overlay)
fig, axes = plt.subplots(4, 4, figsize=(14, 14))

categories = [
    ("True Positive (Correctly identified PNEUMONIA)", tp_samples),
    ("True Negative (Correctly identified NORMAL)", tn_samples),
    ("False Positive (Incorrectly predicted PNEUMONIA)", fp_samples),
    ("False Negative (Incorrectly predicted NORMAL)", fn_samples),
]

for row, (category_name, sample_indices) in enumerate(categories):
    for col, idx in enumerate(sample_indices):
        # Load image and forward pass
        img, label, path = test_ds[idx]
        img_batch = img.unsqueeze(0).to(DEVICE)
        
        # Generate Grad-CAM (targeting PNEUMONIA class)
        with torch.enable_grad():
            heatmap = gradcam.generate(img_batch, class_idx=1)
        
        # Overlay heatmap on image
        overlay = overlay_heatmap(img, heatmap, alpha=0.5, cmap_name="jet")
        
        # Display
        axes[row, col].imshow(overlay)
        true_label = INV_LABEL_MAP[int(label)]
        pred_label = INV_LABEL_MAP[int(y_pred[idx])]
        prob = y_prob[idx]
        axes[row, col].set_title(f"True: {true_label}\nPred: {pred_label} ({prob:.2f})", fontsize=9)
        axes[row, col].axis("off")

# Add row labels
row_labels = [
    "True Positive\n(Correct PNEUMONIA)",
    "True Negative\n(Correct NORMAL)",
    "False Positive\n(Wrong -> PNEUMONIA)",
    "False Negative\n(Wrong -> NORMAL)",
]
for i, label in enumerate(row_labels):
    fig.text(0.02, 0.88 - (i * 0.24), label, va="center", fontsize=10, weight="bold")

plt.suptitle("Grad-CAM Visualizations by Classification Outcome\n(Heatmap highlights regions important for PNEUMONIA prediction)", 
             fontsize=12, weight="bold", y=0.995)
plt.tight_layout(rect=[0.05, 0, 1, 0.99])
plt.savefig(reports_dir / "gradcam_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nGrad-CAM analysis saved to", reports_dir / "gradcam_analysis.png")

# Clean up
gradcam.remove_hooks()